In [16]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict


In [17]:

class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int

    sr: float
    bpb: float
    boundary_percentage: float
    summary: str


In [18]:
def calculate_sr(state: BatsmanState) -> dict:

    sr = (state['runs'] / state['balls']) * 100

    # return only the key this node updates
    return {
        'sr': sr
    }


In [19]:
def calculate_bpb(state: BatsmanState) -> dict:

    boundaries = state['fours'] + state['sixes']

    bpb = state['balls'] / boundaries if boundaries else 0.0

    return {
        'bpb': bpb
    }


In [20]:
def calculate_boundary_percentage(state: BatsmanState) -> dict:

    boundary_percentage = ((state['fours'] * 4 + state['sixes'] * 6) / state['runs']) * 100

    return {
        'boundary_percentage': boundary_percentage
    }


In [21]:
def summary(state: BatsmanState) -> dict:

    summary = f"""
        Strike Rate - {state['sr']} \n 
        Balls per boundary - {state['bpb']} \n 
        Boundary percent - {state['boundary_percentage']}
        """

    return {
        'summary': summary
    }


In [22]:
graph = StateGraph(BatsmanState)

graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percentage', calculate_boundary_percentage)
graph.add_node('summary', summary)


# edges

graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundary_percentage')

graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('calculate_boundary_percentage', 'summary')

graph.add_edge('summary', END)


workflow = graph.compile()


In [23]:
initial_state = {
    'runs' : 100,
    'balls' : 50,
    'fours': 6,
    'sixes': 4
}

final_state = workflow.invoke(initial_state)

print(final_state['summary'])



        Strike Rate - 200.0 
 
        Balls per boundary - 5.0 
 
        Boundary percent - 48.0
        


In [24]:
from langgraph.graph import state


print(final_state)

{'runs': 100, 'balls': 50, 'fours': 6, 'sixes': 4, 'sr': 200.0, 'bpb': 5.0, 'boundary_percentage': 48.0, 'summary': '\n        Strike Rate - 200.0 \n \n        Balls per boundary - 5.0 \n \n        Boundary percent - 48.0\n        '}
